In [1]:
import sys
print(sys.executable)
!{sys.executable} -m pip install findspark

c:\Users\Usuario\AppData\Local\Programs\Python\Python310\python.exe


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Configuración
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, sum, desc

In [3]:
# Spark SQL
! pip install pyspark[sql]
# pandas API on Spark
! pip install pyspark[pandas_on_spark] plotly  # to plot your data, you can install plotly together.
# Spark Connect
! pip install pyspark[connect]


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Pyspark
Python API (conjunto de reglas y protocolos que permite que diferentes aplicaciones se comuniquen entre sí para intercambiar datos y funcionalidades) for Spark

- When Spark transforms data, it does not immediately compute the transformation but plans how to compute later.When actions such as collect() are explicitly called, the computation starts.

Parameters:
- **.appName("MyApp")**  # to name your application
- **.master("local[*]")**  # to run Spark locally with as many worker threads as logical cores on your machine
- **.config("spark.driver.memory", "2g")**  # to set the amount of memory allocated to the driver process

In [ ]:
# Crear una sesión de Spark
# Es igual hacerlo todo en una línea, el "\" es solo para legibilidad, para avisar que se corta el código   
spark = SparkSession.builder \
    .appName("AnalisisVentas") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

# Leer CSV 
df = spark.read.csv("../data/online_retail.csv", header=True, inferSchema=True)

df.printSchema()
df.show(5, truncate=False)

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+-----------------------------------+--------+------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |12/1/10 8:26|2,55     |17850     |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |12/1/10 8:26|3,39     |17850     |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |12/1/10 8:26|2,7

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 55749)
Traceback (most recent call last):
  File "c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\socketserver.py", line 747, in __init__
    self.handle()
  File "c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\pyspark\accumulators.py", line 299, in handle
    poll(accum_updates)
  File "c:\Users\Usuario\AppData\Local\Programs\Python\P

- *printSchema()*: to display the structure of a DataFrame, including column names, data types, and nullability.
- *show()*: to display the contents of a DataFrame in a tabular format.

In [5]:
# Crear dataframe (no cargarlo)
from datetime import datetime, date
import pandas as pd
from pyspark.sql import Row

df_2 = spark.createDataFrame([
    Row(a=1, b=2., c='string1', d=date(2000, 1, 1), e=datetime(2000, 1, 1, 12, 0)),
    Row(a=2, b=3., c='string2', d=date(2000, 2, 1), e=datetime(2000, 1, 2, 12, 0)),
    Row(a=4, b=5., c='string3', d=date(2000, 3, 1), e=datetime(2000, 1, 3, 12, 0))
])
df_2.printSchema()
df_2.show(truncate=False)

root
 |-- a: long (nullable = true)
 |-- b: double (nullable = true)
 |-- c: string (nullable = true)
 |-- d: date (nullable = true)
 |-- e: timestamp (nullable = true)

+---+---+-------+----------+-------------------+
|a  |b  |c      |d         |e                  |
+---+---+-------+----------+-------------------+
|1  |2.0|string1|2000-01-01|2000-01-01 12:00:00|
|2  |3.0|string2|2000-02-01|2000-01-02 12:00:00|
|4  |5.0|string3|2000-03-01|2000-01-03 12:00:00|
+---+---+-------+----------+-------------------+



## Correcciones y validaciones del dataframe

In [6]:
from pyspark.sql.functions import regexp_replace, col, to_timestamp

# Tipo de columnas
df = df.withColumn("UnitPrice", regexp_replace(col("UnitPrice"), ",", ".")) # Reemplazar comas por puntos (separador decimal pyspark)
df = df.withColumn("UnitPrice", col("UnitPrice").cast("double"))
df = df.withColumn("InvoiceDate", to_timestamp("InvoiceDate", "M/d/yy H:mm"))  # Convertir a timestamp

# Valores nulos
# df.na.drop()                        # Eliminar filas con nulos
df = df.na.fill({"Quantity": 0})            # Rellenar nulos
df.filter(df["UnitPrice"].isNull())     # Filas con nulos en una columna


DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: timestamp, UnitPrice: double, CustomerID: int, Country: string]

In [7]:
# Conteo y % de nulos por columna
from pyspark.sql.functions import col, sum as spark_sum, round

total_rows = df.count()

# Convierte nulos en 1, no nulos en 0, los suma y renombra el resultado con el nombre de la columna.
null_summary = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in df.columns
])

# Convertir a un DF de dos filas: nulos absolutos y porcentaje
percent_summary = null_summary.select([
    round((col(c) / total_rows) * 100, 2).alias(c)
    for c in df.columns
])

print("🔹 Cantidad de nulos:")
null_summary.show()

print("🔹 Porcentaje de nulos:")
percent_summary.show()


🔹 Cantidad de nulos:
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|        0|        0|       1454|       0|          0|        0|    135080|      0|
+---------+---------+-----------+--------+-----------+---------+----------+-------+

🔹 Porcentaje de nulos:
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|      0.0|      0.0|       0.27|     0.0|        0.0|      0.0|     24.93|    0.0|
+---------+---------+-----------+--------+-----------+---------+----------+-------+



In [8]:
# Número de filas
print(f"Total de registros: {df.count()}")

# Número de columnas
print(f"Total de columnas: {len(df.columns)}")

# Lista de columnas
print(df.columns)

# Tipo de columnas
print(df.dtypes)

Total de registros: 541909
Total de columnas: 8
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
[('InvoiceNo', 'string'), ('StockCode', 'string'), ('Description', 'string'), ('Quantity', 'int'), ('InvoiceDate', 'timestamp'), ('UnitPrice', 'double'), ('CustomerID', 'int'), ('Country', 'string')]


In [9]:
df.describe().show() # estadísticas descriptivas básicas
# Para strings devuelve count, min, max. Donde el mínimo y el máximo lo define según el orden alfabetico
# no esta definida para date, timestamp o booleanos.
# el .show() es para mostrar el resultado en consola de dataframe

+-------+------------------+------------------+--------------------+------------------+-----------------+------------------+-----------+
|summary|         InvoiceNo|         StockCode|         Description|          Quantity|        UnitPrice|        CustomerID|    Country|
+-------+------------------+------------------+--------------------+------------------+-----------------+------------------+-----------+
|  count|            541909|            541909|              540455|            541909|           541909|            406829|     541909|
|   mean|  559965.752026781|27623.240210938104|             20713.0|  9.55224954743324| 4.61111362608925|15287.690570239585|       NULL|
| stddev|13428.417280796919| 16799.73762842769|                NULL|218.08115785023327|96.75985306117953| 1713.600303321597|       NULL|
|    min|            536365|             10002| 4 PURPLE FLOCK D...|            -80995|        -11062.06|             12346|  Australia|
|    max|           C581569|             

In [10]:
# Limpiar DB dejando solo las ventas positivas
df = df.filter(df.Quantity > 0)

# Limpiar DB eliminando filas con nulos en InvoiceDate
df = df.filter(df.InvoiceDate.isNotNull())


In [11]:
from pyspark.sql.functions import col, count, avg

num_paises = df.select("Country").distinct().count()
print(f"Número de países distintos: {num_paises}")


df.groupBy("Country").count().orderBy(col("count").desc()).show(5)

Número de países distintos: 38
+--------------+------+
|       Country| count|
+--------------+------+
|United Kingdom|486286|
|       Germany|  9042|
|        France|  8408|
|          EIRE|  7894|
|         Spain|  2485|
+--------------+------+
only showing top 5 rows


In [12]:
# Lo mismo que arriba pero más organizado con el salto de línea "\"
df.groupBy("Country").count()\
    .orderBy(col("count")\
    .desc())\
    .show(5)

+--------------+------+
|       Country| count|
+--------------+------+
|United Kingdom|486286|
|       Germany|  9042|
|        France|  8408|
|          EIRE|  7894|
|         Spain|  2485|
+--------------+------+
only showing top 5 rows


## Selección y filtrado

In [13]:
df.select('Country','Quantity').show(3)       # Seleccionar columnas
df.filter(df.Quantity > 100).show(5)        # Filtrar filas
df.where(df.Country == "Spain").show(5) # Igual que filter()
df.distinct().count()                  # Contar valores únicos


+--------------+--------+
|       Country|Quantity|
+--------------+--------+
|United Kingdom|       6|
|United Kingdom|       6|
|United Kingdom|       8|
+--------------+--------+
only showing top 3 rows
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536378|    21212|PACK OF 72 RETROS...|     120|2010-12-01 09:37:00|     0.42|     14688|United Kingdom|
|   536387|    79321|       CHILLI LIGHTS|     192|2010-12-01 09:58:00|     3.82|     16029|United Kingdom|
|   536387|    22780|LIGHT GARLAND BUT...|     192|2010-12-01 09:58:00|     3.37|     16029|United Kingdom|
|   536387|    22779|WOODEN OWLS LIGHT...|     192|2010-12-01 09:58:00|     3.37|     16029|United Kingdom|
|   536387|    22466|FAIRY TALE COTTAG

526054

## Agrupaciones y resúmenes

In [14]:
df.groupBy("Country").count().show(5)                     # Conteo por grupo
df.groupBy("Country").agg(avg("Quantity"), spark_sum("UnitPrice")).show(5)  # Agregaciones múltiples
df.orderBy(desc("Quantity")).show(5)                        # Ordenar descendente


+-------+-----+
|Country|count|
+-------+-----+
| Sweden|  451|
|Germany| 9042|
| France| 8408|
|Belgium| 2031|
|Finland|  685|
+-------+-----+
only showing top 5 rows
+-------+------------------+------------------+
|Country|     avg(Quantity)|    sum(UnitPrice)|
+-------+------------------+------------------+
| Sweden| 80.00665188470066|1695.7900000000002|
|Germany|13.189891616898915|33532.139999999934|
| France| 13.33301617507136| 36992.79000000002|
|Belgium|11.441161989167897| 7372.850000000001|
|Finland|15.626277372262773| 3628.439999999999|
+-------+------------------+------------------+
only showing top 5 rows
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   581483|    23843|PAPER CRAFT , LIT...|   80995|2

In [15]:
# Primera y ultima fecha 
from pyspark.sql.functions import min, max
df.select(min("InvoiceDate"), max("InvoiceDate")).show()


+-------------------+-------------------+
|   min(InvoiceDate)|   max(InvoiceDate)|
+-------------------+-------------------+
|2010-12-01 08:26:00|2011-12-09 12:50:00|
+-------------------+-------------------+



## Operaciones con columnas

Como las funciones select, filter, groupBy, orderBy, withColumn, drop, etc, funcionan con columnas, utilizamos la función lit para crear columnas literales (valor fijo en toda la columna).

El "withColumn" es como el mutate de dplyr en R.

In [16]:
from pyspark.sql.functions import lit, when

#df = df.withColumn("IVA", df.UnitPrice * 0.21)           # Nueva columna
#df = df.withColumnRenamed("Sales", "Ventas")             # Renombrar
#df = df.drop("ColumnaInnecesaria")                       # Eliminar
df = df.withColumn("Etiqueta", when(df.Quantity > 100, "Alta").otherwise("Baja"))


* withColumn: Crear o modificar una columna existente.

In [17]:
# Earnings per country per year
from pyspark.sql.functions import year, month, col # Extraer año, mes, referirse a columna
from pyspark.sql.functions import to_timestamp # Convertir string a timestamp
from pyspark.sql.functions import round, col

df = df.withColumn("Revenue", col("Quantity") * col("UnitPrice")) # Crear columna de ganancias

# Agrupar por año y país, sumar ingresos
df_result = df.groupBy(year("InvoiceDate").alias("Year"), "Country") \
              .agg(spark_sum("Revenue").alias("TotalRevenue"))

df_result = df_result.withColumn("TotalRevenue", round(col("TotalRevenue"), 2))
print(df_result.dtypes)
df_result.orderBy(desc("TotalRevenue")).show(10)

[('Year', 'int'), ('Country', 'string'), ('TotalRevenue', 'double')]
+----+--------------+------------+
|Year|       Country|TotalRevenue|
+----+--------------+------------+
|2011|United Kingdom|  8254828.98|
|2010|United Kingdom|   748268.98|
|2011|   Netherlands|   276661.86|
|2011|          EIRE|    273420.7|
|2011|       Germany|    213626.0|
|2011|        France|    200098.8|
|2011|     Australia|   137488.46|
|2011|         Spain|    59733.38|
|2011|   Switzerland|    55784.98|
|2011|       Belgium|    39386.43|
+----+--------------+------------+
only showing top 10 rows


# Simulación Test

In [25]:
# Leer CSV 
df = spark.read.csv("../data/online_retail.csv", header=True, inferSchema=True)

df.printSchema()
df.show(5, truncate=False)

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+-----------------------------------+--------+------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |12/1/10 8:26|2,55     |17850     |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |12/1/10 8:26|3,39     |17850     |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |12/1/10 8:26|2,7

## Exploración y comprensión de datos

* ¿Qué significa que una columna sea nullable = true en el esquema?

**Rta:** Significa que la variable puede contener valores nulos.

* ¿Qué tipo de variable representa CustomerID? ¿Cómo podrías verificar cuántos valores nulos tiene?

**Rta:** 


- df.select("CustomerID") devuelve un dataframe que contiene solo esa columna.
- df.CustomerID devuelve una columna específica del dataframe (objeto column). Es equivalente a df["CustomerID"].

El segundo puede generar problemas si el nombre de la columna tiene espacios o caracteres especiales.

In [26]:
print(df.select('CustomerID').dtypes) # tiene que ser con el select porque el dtypes es un atributo del dataframe
# numero de valores nulos en CustomerID
print('la variable CustomerID tiene',df.filter(df.CustomerID.isNull()).count(),'valores nulos')

[('CustomerID', 'int')]
la variable CustomerID tiene 135080 valores nulos


* ¿Cómo obtendrías el número total de registros únicos de clientes (CustomerID)?

**Rta:**

In [27]:
df.select('CustomerID').distinct().count()

4373

* ¿Qué diferencia hay entre distinct() y dropDuplicates() en PySpark?

**Rta:** distinct() elimina filas duplicadas considerando todas las columnas del DataFrame, mientras que dropDuplicates() permite especificar columnas particulares para identificar duplicados, eliminando filas que tienen valores repetidos en esas columnas. Si se llama dropDuplicates() sin argumentos, se comporta de forma idéntica a distinct(). 

## Limpieza y transformación

* El campo UnitPrice está como texto y usa coma como separador decimal. Explica paso a paso cómo lo transformarías en una columna numérica (double).

In [28]:
df = df.withColumn('UnitPrice', regexp_replace(col('UnitPrice'), ',', '.'))
df = df.withColumn('UnitPrice', col('UnitPrice').cast('double'))
print(df.select('UnitPrice').dtypes)

[('UnitPrice', 'double')]


* Supón que hay registros con Quantity <= 0. ¿Qué significan y cómo los tratarías?

**Rta:** Estos registros pueden representar devoluciones o errores. Por lo tanto, primero investigaría su origen. Si son devoluciones, crearía una etiqueta para evitar incluirlos erroneamente en análisis de ventas y además poder hacer un análisis de este grupo aposteriori. Si son errores, consideraría eliminarlos o corregirlos según el contexto.

* ¿Cómo crearías una nueva columna llamada TotalValue que sea el producto de Quantity * UnitPrice?

**Rta:**

In [29]:
df = df.withColumn('TotalValue', col('Quantity') * col('UnitPrice'))
df.select('TotalValue').describe().show()

+-------+-----------------+
|summary|       TotalValue|
+-------+-----------------+
|  count|           541909|
|   mean|17.98779487699983|
| stddev|378.8108235059743|
|    min|        -168469.6|
|    max|         168469.6|
+-------+-----------------+



* ¿Cómo convertirías InvoiceDate a tipo fecha para extraer el año y el mes?

In [32]:
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- TotalValue: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)



In [ ]:
# La forma secuencial de la lista es importante. Si pongo col('InvoiceDate') en Year y Month, no funciona porque InvoiceDate no está convertido a timestamp aún.    
df = df.withColumns({'InvoiceDate': to_timestamp('InvoiceDate', 'M/d/yy H:mm'),
                     'Year': year('InvoiceDate'),
                    'Month': month('InvoiceDate')
                     })

df.select('InvoiceDate','Year','Month').show(5)

+-------------------+----+-----+
|        InvoiceDate|Year|Month|
+-------------------+----+-----+
|2010-12-01 08:26:00|2010|   12|
|2010-12-01 08:26:00|2010|   12|
|2010-12-01 08:26:00|2010|   12|
|2010-12-01 08:26:00|2010|   12|
|2010-12-01 08:26:00|2010|   12|
+-------------------+----+-----+
only showing top 5 rows


## Análisis y agregación

* ¿Cómo obtendrías las ventas totales por país?


In [43]:
df.filter(col('Quantity') > 0).groupBy('Country').agg(spark_sum('TotalValue').alias('TotalSales')).show(5)

+-------+------------------+
|Country|        TotalSales|
+-------+------------------+
| Sweden|          38378.33|
|Germany|228867.13999999987|
| France|209715.11000000007|
|Belgium|          41196.34|
|Finland|          22546.08|
+-------+------------------+
only showing top 5 rows


* ¿Y las ventas promedio por producto (StockCode)?

In [44]:
df.filter(col('Quantity') > 0).groupBy('StockCode').agg(avg('TotalValue').alias('AvgSales')).show(5)

+---------+------------------+
|StockCode|          AvgSales|
+---------+------------------+
|    22728| 26.10611601513241|
|    21889|14.648413223140494|
|   90210B| 8.794285714285715|
|    21259|24.762891986062723|
|    21894|6.7785820895522395|
+---------+------------------+
only showing top 5 rows


* ¿Cómo identificarías el país con más transacciones y su volumen total de ventas?

In [54]:
df.filter(col('Quantity') > 0) \
.withColumn("Sales", col("Quantity") * col("UnitPrice")) \
.groupBy('Country')  \
.agg(
    count('InvoiceNo').alias('TotalOrders'), 
    spark_sum('Sales').alias('TotalSales'))\
.orderBy(desc('TotalOrders'))\
.show(5)

+--------------+-----------+------------------+
|       Country|TotalOrders|        TotalSales|
+--------------+-----------+------------------+
|United Kingdom|     486286| 9003097.964000123|
|       Germany|       9042|228867.13999999987|
|        France|       8408|209715.11000000007|
|          EIRE|       7894|283453.96000000037|
|         Spain|       2485| 61577.10999999999|
+--------------+-----------+------------------+
only showing top 5 rows


* Si quisieras analizar la tendencia de ventas mensuales, ¿qué pasos seguirías en PySpark?

**Rta:** Haría un groupBy del mes y año extraidos de InvoiceDate. Luego usaría una función de agregación para calcular las ventas totales por mes. Finalmente, ordenaría los resultados por año y mes para observar la tendencia a lo largo del tiempo.

In [55]:
df.filter(col('Quantity') > 0) \
.withColumn("Sales", col("Quantity") * col("UnitPrice")) \
.groupBy('Year', 'Month') \
.agg(spark_sum('Sales').alias('TotalSales'))\
.orderBy('Year', 'Month')\
.show(12)

+----+-----+------------------+
|Year|Month|        TotalSales|
+----+-----+------------------+
|2010|   12| 823746.1399999646|
|2011|    1| 691364.5600000193|
|2011|    2| 523631.8900000187|
|2011|    3| 717639.3600000187|
|2011|    4| 537808.6210000126|
|2011|    5| 770536.0200000152|
|2011|    6| 761739.9000000219|
|2011|    7| 719221.1910000228|
|2011|    8| 737014.2600000158|
|2011|    9|1058590.1720000212|
|2011|   10| 1154979.300000051|
|2011|   11|1509496.3300000164|
+----+-----+------------------+
only showing top 12 rows


## Casos prácticos

In [ ]:
df.withColumn("Sales", col("Quantity") * col("UnitPrice"))

DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: int, InvoiceDate: timestamp, UnitPrice: double, CustomerID: int, Country: string, TotalValue: double, Year: int, Month: int, Sales: double]

* Imagina que quieres calcular el ticket promedio por cliente (promedio de TotalValue por CustomerID).¿Cómo lo harías?


In [62]:
df.filter(col('Quantity') > 0) \
.groupBy('CustomerID') \
.agg(round(avg('TotalValue'),2).alias('AvgTicket'))\
.show(10)

+----------+---------+
|CustomerID|AvgTicket|
+----------+---------+
|     17420|    19.96|
|     16503|    17.05|
|     15727|    17.15|
|     15100|    292.0|
|     16916|     4.03|
|     12471|     43.1|
|     17809|    88.72|
|     15738|    27.82|
|     17223|     8.71|
|     18043|     4.62|
+----------+---------+
only showing top 10 rows


* ¿Cómo podrías detectar clientes que solo realizaron una compra?

In [67]:
from pyspark.sql.functions import countDistinct

df.filter(col('Quantity') > 0) \
.groupBy('CustomerID') \
.agg(countDistinct('InvoiceNo').alias('NumPurchases')) \
.filter(col('NumPurchases') == 1) \
.show(10)


+----------+------------+
|CustomerID|NumPurchases|
+----------+------------+
|     15447|           1|
|     16339|           1|
|     16574|           1|
|     15957|           1|
|     15619|           1|
|     15790|           1|
|     13832|           1|
|     14536|           1|
|     14423|           1|
|     14148|           1|
+----------+------------+
only showing top 10 rows



* Si quisieras entrenar un modelo de clustering de clientes basándote en sus patrones de compra, ¿qué variables derivadas crearías y por qué?

In [ ]:
# FINALIZAR SESION
#spark.stop()